# Data Description

- Author: RUEI-HSIANG, CHANG（張睿庠，B12303030），AI 協助整理與驗證。
- Date first created: 20261120
- Background: 2026 年縣市首長與臺北、高雄市議員之候選人公報政見、圖片辨識與文字向量。
- Unit of observation: 全文表一列一位候選人，文字段落與跨人最近鄰表一列一段，兩個向量矩陣每列對應一段輸入文字。診斷表每列一組人工設計的測試句，另保存模型紀錄與逐卡 OCR 證據。
- Sample period: 2026 選舉公報。
- Known issues: 設計圖可能含競選口號、過往政績、低解析文字與跨欄錯序。分段與辨識準確率待人工評估。相似度可用於查找文字，研究變數須另編碼。
- Definition for each variable: 姓名、公職、縣市與 candidate_key 表示歸屬。platform_text 為採用的全文，upstream_platform_text 為先前版本。file、page、source_url、card_bbox、card_id 與 source_sha256 定位原始證據。match_method 是上游身分連結方法。ocr_agreement 是兩套辨識中文字序列的 SequenceMatcher 比率。review_priority 在比率低於 0.5 時標為優先人工核對。platform_source 記錄上游來源，selected_reading 記錄本次採用讀法。platform_image_share 為上游估算之圖片占比。bulletin_status、bulletin_note、extraction_review_status 說明缺漏與審核狀態。election_year 為選舉年，text_sha256 為 UTF-8 選用原文指紋。embedding_text 為送入模型的字串，token_count 含特殊符號，embedding_row 為由 0 起算的向量列號。segment_id、segment_number 代表候選人內文字段落，char_start、char_end 表示在 platform_text 的起點及終點（不含）。text_original 保留原段落。topic、subtopic、stance、scope 與 label_status 供後續人工編碼。neighbors 表含對照雙方識別碼與全文及 cosine_similarity。diagnostics 表含 pair_type、text_a、text_b、source 與相似度。OCR JSON 逐卡保存裁切框與每行文字、辨識信心與相對於裁切圖的座標。visual_check.csv 保存沈伯洋公報36項勾選框政見的AI目視轉錄與頁碼。run JSON 記錄模型、版本、環境與各檔指紋。
- Data path / Downstream analysis input / SHA-256: 下方逐檔固定。
- Source data: candidate_bulletin_2026.csv、bulletin_2026_cards_ocr.json 及 cec_bulletin_2026/file_manifest.json 所列原始公報。
- Processing code: [處理筆記本](../code/bulletin_embeddings_2026.ipynb)、[影像辨識](../code/bulletin_ocr_2026.py)。
- Role: 政見檢索與人工編碼的輸入。
- Verification status: 以下 Assertions 從磁碟讀取並核對。

In [1]:
from pathlib import Path
import pandas as pd, numpy as np, hashlib, json
root=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'AGENTS.md').is_file() and (p/'templates').is_dir())
out=root/'data/processed'
stem='bulletin_embeddings_2026_'
def read(n): return pd.read_csv(out/(stem+n+'.csv'),dtype=str,keep_default_na=False)


## Change for Your Own Data

In [2]:
EXPECTED = {'data/processed/bulletin_embeddings_2026_corpus.csv': '28c6654cb7f460ad1b140b581b675dba72d580d37fcdad9364561359d5705d54', 'data/processed/bulletin_embeddings_2026_segments.csv': '1a68f78d441edb77a5bb88c84007b3696cd32176dd2a68b257a278c13acb39fb', 'data/processed/bulletin_embeddings_2026_candidate_vectors.npy': 'befb94644785578ca88a166e2d2702ed2c4fefd9d9df3c8d8b4eaca05df304a3', 'data/processed/bulletin_embeddings_2026_segment_vectors.npy': '2198ca4af2af05878f3c1f90817c3a4aed77cade98a07b1177e603f3ede358ca', 'data/processed/bulletin_embeddings_2026_neighbors.csv': '976d68f296a86d0770247afeecd8421ab01852d2ec48477d93ead87955adc98f', 'data/processed/bulletin_embeddings_2026_diagnostics.csv': '196317a95d9117263820f675b01bf14138b243b9c9ef58bec2ef9aa5a3a7b02a', 'data/processed/bulletin_embeddings_2026_run.json': '171049d6c8571b299f4f72b4a8e097a6ddc5f44df141faf35691a0fba26fc6fa', 'data/processed/bulletin_embeddings_2026_ocr.json': 'b1ee816896056cbd56729a48ab1d0eff671ef97e12014fe97c9e698f272891af', 'data/processed/bulletin_embeddings_2026_visual_check.csv': '3e4cce0c05c693b52150037856b9abd2b5270ae133ff124d0c0d5460970f77f8'}
for path, expected in EXPECTED.items():
    assert hashlib.sha256((root/path).read_bytes()).hexdigest()==expected,path

## Summary

In [3]:
c,s,n,d=[read(k) for k in ['corpus','segments','neighbors','diagnostics']]
v,w=[np.load(out/(stem+k+'_vectors.npy'),allow_pickle=False) for k in ['candidate','segment']]
print(v.shape,w.shape)
print(c.selected_reading.value_counts(dropna=False))


(279, 1024) (3598, 1024)
selected_reading
ocr              120
text_layer        95
tesseract_ocr     64
                   1
Name: count, dtype: int64


### Sample

In [4]:
s[['candidate_name','segment_number','embedding_text','token_count']].head(10)

,candidate_name,segment_number,embedding_text,token_count
0,郭璽,1,一. 活化商圈、夜間經濟、觀光、會展與文化產業，引進國際企業及創新產業，創造青年就 業與創業...,41
1,郭璽,2,二. 建構15分鐘高齡生活圈，整合醫療、長照、共餐、交通與住宅服務；,28
2,郭璽,3,強化獨居長者關懷 及高齡友善住宅，使長者安心生活。,20
3,郭璽,4,三. 推動高齡族群活化生活政策，結合社區關懷據點、市民活動中心及長者服務據點，規劃 多元益智...,47
4,郭璽,5,四. 推動老屋更新、危老重建及都市更新行政流程改革，建立跨局處單一窗口、縮短審議與 補件時間；,38
5,郭璽,6,同步強化老屋耐震、電梯及無障礙改善。,16
6,郭璽,7,五. 建立「台北國際智慧城市創新基地」，串聯交通、醫療、長照、觀光、環境及公共建設 等城市資...,86
7,郭璽,8,六. 推動「新舊社區共融升級計畫」，老社區優先改善道路、人行空間、老舊管線、停車、 無障礙設...,41
8,郭璽,9,新興社區則導入智慧交通、充電設施、綠能管理、數位服務與 公共空間整合，使其具備下一世代城市標...,51
9,郭璽,10,七. 推動「跨世代文化共融計畫」，重新活化市民活動中心、圖書館、市場、公園及閒置公 共空間，...,71


### Summary Stats

In [5]:
print(s.groupby('candidate_key').size().describe())
print(n.cosine_similarity.astype(float).describe())
d

count    279.000000
mean      12.896057
std       11.187735
min        1.000000
25%        5.000000
50%       10.000000
75%       17.000000
max       65.000000
dtype: float64
count    3598.000000
mean        0.708486
std         0.066128
min         0.441744
25%         0.666791
50%         0.710123
75%         0.751880
max         1.000000
Name: cosine_similarity, dtype: float64


,pair_type,text_a,text_b,cosine_similarity,source
0,同議題同立場,支持重啟核電廠。,贊成恢復核能發電。,0.920354425907135,synthetic_diagnostic_not_bulletin
1,同議題反立場,支持重啟核電廠。,反對重啟核電廠。,0.8470155000686646,synthetic_diagnostic_not_bulletin
2,不同議題,支持重啟核電廠。,增設公共托育中心。,0.5828487873077393,synthetic_diagnostic_not_bulletin
3,同議題不同措施,興建社會住宅。,擴大租金補貼。,0.6751764416694641,synthetic_diagnostic_not_bulletin
4,同議題不同尺度,改善本市公共運輸。,改革全國公共運輸補助制度。,0.7932101488113403,synthetic_diagnostic_not_bulletin


### Assertions

In [6]:
assert c.candidate_key.is_unique and s.segment_id.is_unique
assert set(c.election_year)=={'2026'}
available=c[c.embedding_row.ne('')]
assert np.array_equal(available.embedding_row.astype(int),np.arange(len(v)))
assert np.array_equal(s.embedding_row.astype(int),np.arange(len(w)))
assert available.candidate_key.to_list()==c.loc[c.platform_text.ne(''),'candidate_key'].to_list()
source=c.set_index('candidate_key')
for t in s.itertuples():
    assert source.at[t.candidate_key,'platform_text'][int(t.char_start):int(t.char_end)]==t.text_original
    assert s.loc[int(t.embedding_row),'segment_id']==t.segment_id
assert set(n.segment_id)==set(s.segment_id)
assert n.candidate_key.ne(n.neighbor_candidate_key).all()
lookup=dict(zip(s.segment_id,s.embedding_row.astype(int)))
left=n.segment_id.map(lookup).to_numpy()
right=n.neighbor_segment_id.map(lookup).to_numpy()
assert np.allclose(np.einsum('ij,ij->i',w[left],w[right]),n.cosine_similarity.astype(float),atol=2e-5)
for matrix in (v,w):
    assert matrix.shape[1]==1024 and matrix.dtype==np.float32
    assert np.isfinite(matrix).all() and np.allclose(np.linalg.norm(matrix,axis=1),1,atol=1e-5)
manual=read('visual_check')
assert len(manual)==36 and manual.candidate_key.nunique()==1
assert set(manual.candidate_key).issubset(set(c.candidate_key))
print('來源版本、段落範圍、向量列、最近鄰相似度與正規化驗證通過。')

來源版本、段落範圍、向量列、最近鄰相似度與正規化驗證通過。
